# When near stops meaning near
Lecture 4 · dimensionality. One story in four parts: **you are a point** → **ten years apart** → **why: the geometry** → **escaping the curse**.
**Predict before every run.** Write your guess down.

In [ ]:
# L4.00 Setup
import os, itertools
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from math import gamma, pi
from scipy.spatial.distance import pdist

rng = np.random.default_rng(0)
LEC, STU = '#D55E00', '#0072B2'   # lecturer, student
ACT = {'downstairs': '#E69F00', 'upstairs': '#D55E00', 'walking': '#CC79A7',
       'jogging': '#7B3294', 'sitting': '#56B4E9', 'standing': '#0072B2'}

try:
    import ipywidgets as w
    def play(f, **s):
        ctl = {}
        for k, v in s.items():
            if isinstance(v, list):
                ctl[k] = w.Dropdown(options=v)
            elif isinstance(v[3], float):
                ctl[k] = w.FloatSlider(min=v[0], max=v[1], step=v[2], value=v[3])
            else:
                ctl[k] = w.IntSlider(min=v[0], max=v[1], step=v[2], value=v[3])
        w.interact(f, **ctl)
except ImportError:  # no sliders: run once with the defaults
    def play(f, **s):
        f(**{k: (v[0] if isinstance(v, list) else v[3]) for k, v in s.items()})

def ball_share(n):
    # share of the cube [-1, 1]^n filled by the unit ball
    return pi ** (n / 2) / gamma(n / 2 + 1) / 2 ** n

# Part 1 · You are a point
Every measurement is an axis. A person is one row of the table, and one point in space.

## 1.1 One person, one point
**Predict:** a clinic records age, resting heart rate and blood pressure. How many dimensions is that? How many does a smartphone give us?

In [ ]:
# L4.01 One person, one point
N = 500
clinic = pd.DataFrame({'age': rng.uniform(20, 80, N), 'resting_hr': rng.normal(70, 10, N)})
clinic['systolic_bp'] = 100 + 0.5 * clinic.age + rng.normal(0, 12, N)   # rises with age
lecturer = dict(age=35, resting_hr=68, systolic_bp=121)
student = dict(age=25, resting_hr=64, systolic_bp=114)
print(pd.DataFrame([lecturer, student], index=['lecturer', 'student']))

fig = plt.figure(figsize=(14, 4))
a = fig.add_subplot(131); a.scatter(clinic.age, rng.uniform(-.05, .05, N), s=6, alpha=.3, color='grey')
a.set_ylim(-.3, .3); a.set_yticks([]); a.set_xlabel('age'); a.set_title('1-D')
b = fig.add_subplot(132); b.scatter(clinic.age, clinic.resting_hr, s=6, alpha=.3, color='grey')
b.set_xlabel('age'); b.set_ylabel('resting HR'); b.set_title('2-D')
c = fig.add_subplot(133, projection='3d'); c.scatter(clinic.age, clinic.resting_hr, clinic.systolic_bp, s=4, alpha=.3, color='grey')
c.set_xlabel('age'); c.set_ylabel('HR'); c.set_zlabel('SBP'); c.set_title('3-D')
for p, col in [(lecturer, LEC), (student, STU)]:
    a.scatter(p['age'], 0, marker='*', s=250, color=col, zorder=3)
    b.scatter(p['age'], p['resting_hr'], marker='*', s=250, color=col, zorder=3)
    c.scatter(p['age'], p['resting_hr'], p['systolic_bp'], marker='*', s=250, color=col)
plt.show()

## 1.2 Every new dimension doubles the corners
**Predict:** a square has 4 corners, a cube 8. How many does a 30-dimensional cube have?

In [ ]:
# L4.02 Corners
fig, axs = plt.subplots(1, 6, figsize=(16, 3))
for n, ax in zip(range(1, 7), axs):
    V = np.array(list(itertools.product([0, 1], repeat=n)), float)
    ang = np.pi * np.arange(n) / max(n, 2) + (0.13 * np.arange(n) if n > 2 else 0)
    rad = 1 + (0.12 * np.arange(n) if n > 2 else 0 * np.arange(n))
    P = V @ (rad[:, None] * np.c_[np.cos(ang), np.sin(ang)])
    for i, j in itertools.combinations(range(len(V)), 2):
        k = np.nonzero(V[i] != V[j])[0]
        if len(k) == 1:
            ax.plot(*P[[i, j]].T, color=plt.cm.viridis(k[0] / 5), lw=1.2)
    ax.scatter(*P.T, s=14, color=LEC, zorder=3)
    ax.set_aspect('equal'); ax.axis('off'); ax.set_title(f'{n}-D: {2 ** n} corners')
plt.show()
print(f'30-D: {2 ** 30:,} corners')

**→ Next:** if space grows this fast, what happens to the distance between two people?

# Part 2 · Ten years apart
The lecturer is 35, a student 25. Clinically, 10 years matter. Do they still matter to a distance?

## 2.1 Where do 10 years matter?
**Predict:** max heart rate ≈ 220 − age (a common rule of thumb). How far apart are the lecturer's and the student's training zones?

In [ ]:
# L4.03 Heart-rate training zones
people = [('Rehab patient, 65', 65, '#4D4D4D'), ('Lecturer, 35', 35, LEC), ('Student, 25', 25, STU)]
zones = [(.5, .6, '#DCEAF7'), (.6, .7, '#A9CFEA'), (.7, .8, '#F9D58B'), (.8, .9, '#F4A261'), (.9, 1.0, '#E76F51')]
fig, ax = plt.subplots(figsize=(12, 3.4))
for k, (name, a, col) in enumerate(people):
    mx = 220 - a
    for lo, hi, zc in zones:
        ax.barh(k, (hi - lo) * mx, left=lo * mx, color=zc, edgecolor='white', height=.6)
    ax.text(.5 * mx - 3, k, name, ha='right', va='center', color=col, fontweight='bold')
    ax.text(mx + 2, k, f'max ≈ {mx} bpm', va='center', color=col, fontweight='bold')
ax.set_xlim(40, 240); ax.set_yticks([]); ax.set_xlabel('heart rate (bpm) · zones = 50–100% of max HR')
plt.show()

## 2.2 How much does the gap count?
**Predict:** with age plus 29 other standardised measurements, what share of the distance between lecturer and student comes from the 10 years?

In [ ]:
# L4.04 Share of the distance from the age gap
D = 30
age_z = (clinic.age - clinic.age.mean()) / clinic.age.std()
Z = np.c_[age_z, rng.standard_normal((N, D - 1))]   # age + 29 other measurements, standardised
gap = 10 / clinic.age.std()                           # 10 years in standard deviations

def gap_share(dims=30):
    d = np.arange(1, dims + 1)
    share = gap ** 2 / (gap ** 2 + 2 * (d - 1))       # a random pair differs by 2 per extra dimension, on average
    plt.figure(figsize=(10, 3.2)); plt.bar(d, share * 100, color=LEC)
    plt.xlabel('dimensions'); plt.ylabel('% of squared distance')
    plt.title(f'Age gap: 100% of the distance in 1-D, {share[-1] * 100:.1f}% in {dims}-D'); plt.show()

play(gap_share, dims=(2, 30, 1, 30))

## 2.3 My nearest neighbours lose my age
**Predict:** among 500 patients, who are the lecturer's 10 most similar people in 1-D? In 30-D?

In [ ]:
# L4.05 Nearest neighbours
me = np.r_[(35 - clinic.age.mean()) / clinic.age.std(), rng.standard_normal(D - 1)]
dims = [1, 2, 3, 5, 10, 20, 30]
fig, (a, b) = plt.subplots(1, 2, figsize=(14, 4), gridspec_kw={'width_ratios': [1.5, 1]})
for i, d in enumerate(dims):
    dist = np.linalg.norm(Z[:, :d] - me[:d], axis=1)
    a.scatter(clinic.age.values[np.argsort(dist)[:10]], np.full(10, i), color=STU)
a.axvline(35, color=LEC, lw=2); a.set_yticks(range(len(dims)), [f'{d}-D' for d in dims]); a.invert_yaxis()
a.set_xlim(20, 80); a.set_xlabel('age of the 10 nearest neighbours'); a.set_title('Who is most similar to the lecturer (35)?')
ratio = []
for d in range(1, D + 1):
    dist = np.linalg.norm(Z[:, :d] - me[:d], axis=1); ratio.append(dist.max() / dist.min())
b.semilogy(range(1, D + 1), ratio, 'o-', color=LEC); b.set_xlabel('dimensions'); b.set_title('farthest ÷ nearest person')
plt.show()

## 2.4 What it costs: k-NN predicts the wrong heart rate
**Predict:** max heart rate depends only on age. Predict it from the 10 most similar patients. Does adding measurements help?

In [ ]:
# L4.06 k-NN prediction error
max_hr = 220 - clinic.age.values + rng.normal(0, 8, N)

def knn_error(d, k=10):
    X = Z[:, :d]; Dm = np.linalg.norm(X[:, None] - X[None], axis=2); np.fill_diagonal(Dm, np.inf)
    nn = np.argsort(Dm, axis=1)[:, :k]
    return np.mean(np.abs(max_hr[nn].mean(1) - max_hr))

err = [knn_error(d) for d in range(1, D + 1)]
plt.figure(figsize=(10, 3.4)); plt.plot(range(1, D + 1), err, 'o-', color=LEC, label='k-NN, k = 10')
plt.axhline(np.mean(np.abs(max_hr - max_hr.mean())), ls='--', color='grey', label='guess the average')
plt.ylim(0, None); plt.xlabel('measurements used (only age is informative)'); plt.ylabel('average error (bpm)'); plt.legend(); plt.show()
print(f'error with age only: {err[0]:.1f} bpm · with 30 measurements: {err[-1]:.1f} bpm')

**→ Next:** why does every extra measurement push people apart? Part 3 looks at the geometry.

# Part 3 · Why: the geometry of many dimensions

## 3.1 Nobody is average
**Predict:** 4,063 US Air Force pilots, 10 body measurements (Daniels, 1952). How many are average (within ±0.3 SD) on all 10?

In [ ]:
# L4.07 The average pilot
daniels = [4063, 1055, 302, 143, 73, 28, 12, 6, 3, 2, 0]  # Daniels 1952, real counts

def pilots(correlation=0.8):
    shared = rng.standard_normal((4063, 1))
    X = np.sqrt(correlation) * shared + np.sqrt(1 - correlation) * rng.standard_normal((4063, 10))
    left = [4063] + list(np.cumprod(np.abs(X) < 0.3, axis=1).sum(0))
    plt.figure(figsize=(9, 3.4))
    plt.plot(range(11), daniels, 'ko-', label='Daniels 1952, real pilots')
    plt.plot(range(11), left, 's--', color=LEC, label=f'simulated, correlation {correlation:.2f}')
    plt.yscale('symlog', linthresh=1); plt.xlabel('measurements checked'); plt.ylabel('pilots still average')
    plt.legend(); plt.show()

play(pilots, correlation=(0.0, 0.95, 0.05, 0.8))

## 3.2 The ball in the box
**Predict:** what share of a cube does the biggest ball inside it fill? In 2-D? In 10-D?

In [ ]:
# L4.08 Darts in a cube
def darts(n=10):
    X = rng.uniform(-1, 1, (4000, n))
    r = np.sqrt((X ** 2).sum(1)); inside = r <= 1
    fig, (a, b, c) = plt.subplots(1, 3, figsize=(15, 4))
    a.scatter(X[:, 0], X[:, 1] if n > 1 else rng.uniform(-1, 1, 4000), s=3, c=np.where(inside, STU, LEC))
    a.add_patch(plt.Circle((0, 0), 1, fill=False)); a.set_aspect('equal'); a.set_title(f'n = {n}: {inside.mean():.1%} inside (exact {ball_share(n):.2%})')
    b.hist(r, bins=40, color='grey'); b.axvline(1, color='k', ls='--'); b.set_title('distance from centre (ball edge = 1)')
    dims = np.arange(1, 21); c.semilogy(dims, [ball_share(d) for d in dims], 'o-', color=STU)
    c.semilogy(n, ball_share(n), 'o', ms=12, mfc='none', mec=LEC); c.set_xlabel('dimensions'); c.set_title('ball ÷ cube')
    plt.show()

play(darts, n=(1, 20, 1, 10))

## 3.3 Spheres in the corners
**Predict:** a box of side 4 holds a unit sphere in each corner; a middle sphere touches them all. When does it poke out of the box?

In [ ]:
# L4.09 The middle sphere
dims = np.arange(1, 21)
plt.figure(figsize=(9, 3.2)); plt.plot(dims, np.sqrt(dims) - 1, 'o-', color=LEC, label='middle sphere radius = √n − 1')
plt.axhline(2, color='k', ls='--', label='box wall'); plt.xlabel('dimensions'); plt.legend(); plt.show()

## 3.4 All peel, no orange · the soap bubble
**Predict:** in a 60-D orange, what share is the outer 5%? Among 1,000 patients with 100 features, how many are close to the average patient?

In [ ]:
# L4.10 Peel and bubble
def peel_bubble(n=60):
    fig, (a, b) = plt.subplots(1, 2, figsize=(14, 3.4))
    a.hist(rng.uniform(size=5000) ** (1 / n), bins=50, range=(0, 1), color='grey'); a.axvspan(.95, 1, color='orange', alpha=.3)
    a.set_title(f'uniform in a {n}-D ball: {1 - .95 ** n:.0%} in the outer 5%'); a.set_xlabel('distance from centre')
    d = np.linalg.norm(rng.standard_normal((1000, n)), axis=1)
    b.hist(d, bins=50, color=STU); b.axvline(np.sqrt(n), color='k', ls='--'); b.set_xlim(0, None)
    b.set_title(f'{n} bell-curve features: closest to average at {d.min():.1f}'); b.set_xlabel('distance to the average patient')
    plt.show()

play(peel_bubble, n=(1, 200, 1, 60))

## 3.5 Every pair the same distance
**Predict:** sketch the histogram of all pairwise distances in 2-D, then in 1000-D.

In [ ]:
# L4.11 Pairwise distances
fig, axs = plt.subplots(1, 4, figsize=(16, 3.2), sharey=True)
for ax, n in zip(axs, [2, 10, 100, 1000]):
    d = pdist(rng.random((300, n)))
    ax.hist(d / d.mean(), bins=60, range=(0, 2), color=STU)
    ax.set_title(f'{n}-D: mean {d.mean():.2f}, SD {d.std():.2f}'); ax.set_xlabel('distance ÷ mean')
plt.show()

## 3.6 Nearest versus farthest · everyone at right angles · neighbourhoods that are not local
**Predict:** in 1000-D, how much farther is your farthest neighbour than your nearest? What angle separates two random patients? How wide must a box be to hold your 10% most similar patients in 10-D?

In [ ]:
# L4.12 Three faces of the same effect
fig, (a, b, c) = plt.subplots(1, 3, figsize=(16, 3.6))
dims = [1, 2, 3, 5, 10, 20, 60, 100, 200, 500, 1000]
for name, p in [('L1', 1), ('L2', 2), ('L∞', np.inf)]:
    ratio = []
    for n in dims:
        d = np.linalg.norm(rng.random((500, n)) - rng.random(n), ord=p, axis=1); ratio.append(d.max() / d.min())
    a.loglog(dims, ratio, 'o-', label=name)
a.axvline(60, color='grey', ls=':'); a.set_xlabel('dimensions'); a.set_title('farthest ÷ nearest'); a.legend()
for n in [2, 10, 100, 1000]:
    A, B = rng.standard_normal((2, 3000, n))
    cos = (A * B).sum(1) / np.linalg.norm(A, axis=1) / np.linalg.norm(B, axis=1)
    b.hist(np.degrees(np.arccos(cos)), bins=90, range=(0, 180), histtype='step', lw=2, label=f'{n}-D')
b.set_xlabel('angle between two random patients (°)'); b.legend()
p = np.arange(1, 61)
for share in (.01, .1):
    c.plot(p, share ** (1 / p), label=f'{share:.0%} of patients')
c.set_ylim(0, 1.05); c.set_xlabel('features'); c.set_title('box width needed (share of each range)'); c.legend()
plt.show()

**→ Next:** real data are not random. Part 4 shows what protects us, and what to do about it.

# Part 4 · Escaping the curse

## 4.1 Structure saves you, noise does not
**Predict:** 60 features driven by 2 hidden factors. Add 500 noise features. Do neighbours survive?

In [ ]:
# L4.13 Hidden factors plus noise
def nearest(M):
    Dm = np.linalg.norm(M[:, None] - M[None], axis=2); np.fill_diagonal(Dm, np.inf)
    return Dm.argmin(1)

def structure(factors=2, noise_features=500):
    Zf = rng.standard_normal((240, factors))
    S = Zf @ rng.standard_normal((factors, 60)) + 0.1 * rng.standard_normal((240, 60))
    S = (S - S.mean(0)) / S.std(0)
    X = np.hstack([S, rng.standard_normal((240, noise_features))])
    kept = np.mean(nearest(S) == nearest(X)); d = pdist(X)
    plt.figure(figsize=(9, 3.2)); plt.hist(d / d.mean(), bins=60, range=(0, 2), color=STU)
    plt.xlabel('distance ÷ mean'); plt.title(f'{X.shape[1]} columns: nearest neighbour kept for {kept:.0%} of points'); plt.show()

play(structure, factors=(1, 20, 1, 2), noise_features=(0, 1000, 50, 500))

## 4.2 Our data: how many dimensions do 60 features really use?
**Predict:** how many principal components keep 80% of the variance of the 60 smartphone features?

In [ ]:
# L4.14 Load the course features (from tutorial 2)
# In Colab, first: from google.colab import drive; drive.mount('/content/drive')
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
paths = ['/content/drive/MyDrive/Colab Data/ai_in_rehab_2026_tutorial/features_df_trial.csv',
         '../dataset/Tutorial-MotionSense/Features/features_df_trial.csv']
path = next((p for p in paths if os.path.exists(p)), None)
assert path, 'Run tutorial 2 first: it saves features_df_trial.csv to your Drive.'
df = pd.read_csv(path)
feats = [c for c in df.columns if c not in ('subject', 'activity', 'trial')]
X, y, groups = df[feats].values, df.activity.values, df.subject.values
Xs = StandardScaler().fit_transform(X)
c_act = [ACT[a] for a in y]
print(df.shape[0], 'trials ·', df.subject.nunique(), 'participants ·', len(feats), 'features')

In [ ]:
# L4.15 Explained variance
pca = PCA().fit(Xs); evr = pca.explained_variance_ratio_; cum = np.cumsum(evr)
plt.figure(figsize=(11, 3.4)); plt.bar(range(1, 31), evr[:30] * 100, color='#9BB7D4'); plt.plot(range(1, 31), cum[:30] * 100, 'o-', color=STU)
for t in (80, 90): plt.axhline(t, ls=':', color='grey')
plt.title(f'PC1 {evr[0]:.0%} · 80% needs {np.searchsorted(cum, .8) + 1} components · 90% needs {np.searchsorted(cum, .9) + 1}')
plt.xlabel('component'); plt.ylabel('explained variance (%)'); plt.show()

## 4.3 Scale first, then colour by activity and by participant
**Predict:** without standardising, which feature will PC1 follow?

In [ ]:
# L4.16 Scaling and colouring
raw = PCA(2).fit(X); Zr = raw.transform(X); Zs = PCA(2).fit_transform(Xs)
fig, axs = plt.subplots(1, 3, figsize=(16, 4))
axs[0].scatter(*Zr.T, c=c_act, s=10); axs[0].set_title(f'raw: PC1 {raw.explained_variance_ratio_[0]:.0%} = {feats[np.argmax(abs(raw.components_[0]))]}')
axs[1].scatter(*Zs.T, c=c_act, s=10); axs[1].set_title('standardised, by activity')
axs[2].scatter(*Zs.T, c=pd.factorize(groups)[0], cmap='tab20b', s=10); axs[2].set_title('standardised, by participant')
for a in axs: a.set_xlabel('PC1'); a.set_ylabel('PC2')
plt.show()

## 4.4 t-SNE: the settings change the picture
**Predict:** will perplexity 5 and perplexity 100 show the same clusters?

In [ ]:
# L4.17 t-SNE perplexity
from sklearn.manifold import TSNE
fig, axs = plt.subplots(1, 4, figsize=(16, 3.6))
for ax, per in zip(axs, [5, 30, 50, 100]):
    E = TSNE(perplexity=per, random_state=0, init='pca').fit_transform(Xs)
    ax.scatter(*E.T, c=c_act, s=8); ax.set_title(f'perplexity {per}'); ax.set_xticks([]); ax.set_yticks([])
plt.show()

## 4.5 Five classifiers, tested on new participants
**Predict:** with 2 PCs or with all 60 features, which model wins? Does a random split flatter it?

In [ ]:
# L4.18 Classifier comparison (split by participant)
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupKFold, StratifiedKFold, cross_val_score
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
models = {'Naive Bayes': GaussianNB(), 'k-NN (k=5)': KNeighborsClassifier(5), 'Linear SVM': SVC(kernel='linear'),
          'Polynomial SVM': SVC(kernel='poly', degree=3), 'RBF SVM': SVC()}
by_person = lambda pipe: cross_val_score(pipe, X, y, cv=GroupKFold(6), groups=groups).mean()
rows = []
for name, m in models.items():
    rows.append({'model': name,
                 '2 PCs, new people': by_person(make_pipeline(StandardScaler(), PCA(2), m)),
                 '60 features, new people': by_person(make_pipeline(StandardScaler(), m)),
                 '60 features, random split': cross_val_score(make_pipeline(StandardScaler(), m), X, y,
                                                              cv=StratifiedKFold(6, shuffle=True, random_state=0)).mean()})
print(pd.DataFrame(rows).set_index('model').round(2).to_string())

In [ ]:
# L4.19 The boundaries in 2 PCs (trained on 18 people; dots = 6 new people)
from sklearn.model_selection import GroupShuffleSplit
from matplotlib.colors import ListedColormap
tr, te = next(GroupShuffleSplit(1, test_size=6, random_state=3).split(X, y, groups))
pre = make_pipeline(StandardScaler(), PCA(2)).fit(X[tr]); Ptr, Pte = pre.transform(X[tr]), pre.transform(X[te])
idx = {a: i for i, a in enumerate(ACT)}
lo, hi = np.minimum(Ptr.min(0), Pte.min(0)) - 1, np.maximum(Ptr.max(0), Pte.max(0)) + 1
gx, gy = np.meshgrid(np.linspace(lo[0], hi[0], 300), np.linspace(lo[1], hi[1], 300))
fig, axs = plt.subplots(1, 5, figsize=(18, 3.4))
for ax, (name, m) in zip(axs, models.items()):
    m.fit(Ptr, y[tr]); G = np.vectorize(idx.get)(m.predict(np.c_[gx.ravel(), gy.ravel()])).reshape(gx.shape)
    ax.contourf(gx, gy, G, levels=np.arange(-.5, 6), cmap=ListedColormap(list(ACT.values())), alpha=.3)
    ax.scatter(*Pte.T, c=[ACT[v] for v in y[te]], s=12, edgecolors='k', linewidths=.3)
    ax.set_title(f'{name}: {np.mean(m.predict(Pte) == y[te]):.0%}'); ax.set_xticks([]); ax.set_yticks([])
plt.show()

## Exit check
1. In 30 measurements, what share of the distance between you and someone 10 years older comes from age?
2. Why did k-NN predict max heart rate worse with more measurements, although the truth never changed?
3. What share of a 6-dimensional cube does its inscribed ball fill?
4. The table above: why do 2 PCs score so much lower than 60 features, and why does the random split look better?